# `collabgraph` tutorial

This notebook walks end-to-end through the `collabgraph` package:

1. Read `data/collaborators.xlsx` and inspect the data.
2. Open the embedded Kùzu database and create the schema (node + relationship tables).
3. Ingest the rows with idempotent `MERGE` writes.
4. Verify the graph with a few read-only Cypher queries.
5. Build a NetworkX graph and render customized static visualizations.

**Prerequisites**
- `uv sync` has been run from the project root (so the `collabgraph` package is importable).
- The embedded Kùzu database lives in a single directory (default `./data/collabgraph.kuzu/`); no external server is required.

## 1. Setup

Make the project root the current working directory so that the relative path `data/collaborators.xlsx` and the project-local `.env` are picked up automatically.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

# Run everything from the project root, regardless of where the kernel started.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)

DATA_PATH = PROJECT_ROOT / "data" / "collaborators.xlsx"
OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "_outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Excel path:  ", DATA_PATH)
print("Outputs dir: ", OUTPUT_DIR)


## 2. Read and inspect the input file

`read_collaborators` validates required columns, strips whitespace from string fields, coerces `latitude` / `longitude` to numeric, and drops rows missing required values.

In [ ]:
from collabgraph import read_collaborators

df = read_collaborators(DATA_PATH)
print(f"rows = {len(df)}, cols = {list(df.columns)}")
df.head()


In [ ]:
summary = {
    "unique_collaborators": df["collaborator"].nunique(),
    "unique_sectors": df["sector"].nunique(),
    "unique_affiliations": df["affiliation"].nunique(),
}
summary

## 3. Open the embedded database and ensure schema

The database lives in a single directory on disk (default `./data/collabgraph.kuzu/`). Override it by setting `COLLABGRAPH_DB_PATH` in `.env`.

`init_schema()` creates the `Collaborator`, `Sector`, and `Affiliation` node tables plus the `AFFILIATED_WITH`, `WORKS_IN`, and `PRESENT_AT` relationship tables. It is safe to run repeatedly.

In [ ]:
from collabgraph import Ingestor, load_settings

settings = load_settings()
print(f"db_path={settings.db_path_absolute}")

with Ingestor(settings.db_path) as ing:
    ing.verify_connectivity()
    ing.init_schema()

print("Connected and schema ensured.")

## 4. Ingest the data

All writes use `MERGE`, so re-running this cell creates-or-updates nodes/relationships rather than duplicating them.

The graph schema produced is:

```
(:Collaborator {name})-[:AFFILIATED_WITH]->(:Affiliation {name, address, latitude, longitude, crs})
(:Collaborator)-[:WORKS_IN]->(:Sector {name})
(:Sector)-[:PRESENT_AT]->(:Affiliation)
```

In [ ]:
with Ingestor(settings.db_path) as ing:
    n_written = ing.ingest(df)

print(f"Wrote {n_written} row(s) to Kùzu at {settings.db_path_absolute}.")

### Idempotency check

Running `ingest` again should leave the node and relationship counts unchanged.

In [ ]:
def graph_counts() -> dict[str, int]:
    counts: dict[str, int] = {}
    with Ingestor(settings.db_path) as ing:
        counts["collaborators"] = int(
            ing.conn.execute("MATCH (c:Collaborator) RETURN count(c) AS n").get_as_df().iloc[0, 0]
        )
        counts["sectors"] = int(
            ing.conn.execute("MATCH (s:Sector) RETURN count(s) AS n").get_as_df().iloc[0, 0]
        )
        counts["affiliations"] = int(
            ing.conn.execute("MATCH (a:Affiliation) RETURN count(a) AS n").get_as_df().iloc[0, 0]
        )
        rel_total = 0
        for rel in ("AFFILIATED_WITH", "WORKS_IN", "PRESENT_AT"):
            rel_total += int(
                ing.conn.execute(f"MATCH ()-[r:{rel}]->() RETURN count(r) AS n")
                .get_as_df()
                .iloc[0, 0]
            )
        counts["relationships"] = rel_total
    return counts


before = graph_counts()
with Ingestor(settings.db_path) as ing:
    ing.ingest(df)
after = graph_counts()

print("before:", before)
print("after :", after)
assert before == after, "MERGE-based ingest should be idempotent"
print("Idempotency confirmed.")

## 5. Query the graph with Cypher

Kùzu speaks Cypher. The `Ingestor` exposes its underlying connection at `ing.conn`; use `conn.execute(query, params)` and `.get_as_df()` to get a DataFrame back.

In [ ]:
# Removed: named Cypher examples were dropped in the embedded Kuzu version.
# Use the web Graph/Map views or service helpers instead.


In [ ]:
# Removed: named Cypher examples were dropped in the embedded Kuzu version.
# Use the web Graph/Map views or service helpers instead.


In [ ]:
# Removed: named Cypher examples were dropped in the embedded Kuzu version.
# Use the web Graph/Map views or service helpers instead.


In [ ]:
# Removed: named Cypher examples were dropped in the embedded Kuzu version.
# Use the web Graph/Map views or service helpers instead.


In [ ]:
# Removed: named Cypher examples were dropped in the embedded Kuzu version.
# Use the web Graph/Map views or service helpers instead.


In [ ]:
# Removed: named Cypher examples were dropped in the embedded Kuzu version.
# Use the web Graph/Map views or service helpers instead.


## 6. Build a NetworkX graph and render static visualizations

`build_networkx_graph` turns the same DataFrame into a `MultiDiGraph` whose nodes carry a `kind` attribute (`Collaborator`, `Sector`, `Affiliation`) used by `draw_graph` for styling.

In [ ]:
%matplotlib inline

from collabgraph import build_networkx_graph, draw_graph

g = build_networkx_graph(df)
print(f"nodes = {g.number_of_nodes()}, edges = {g.number_of_edges()}")
kinds = {data["kind"] for _, data in g.nodes(data=True)}
rels = {d["rel"] for _, _, d in g.edges(data=True)}
print(f"node kinds         : {sorted(kinds)}")
print(f"relationship types : {sorted(rels)}")

### 6a. Default rendering

In [ ]:
fig = draw_graph(
    g,
    layout="spring",
    seed=7,
    figsize=(11, 8),
    title="Collaborator network (defaults)",
    output=OUTPUT_DIR / "01_default.png",
)
fig

### 6b. Customized colors, sizes, and edge labels

All styling is overridable. Below we use a different palette, larger affiliation nodes, the `kamada_kawai` layout, and turn on relationship-type labels.

In [ ]:
fig = draw_graph(
    g,
    layout="kamada_kawai",
    node_colors={
        "Collaborator": "#5B8DEF",
        "Sector": "#E07A5F",
        "Affiliation": "#3D8B72",
    },
    node_sizes={
        "Collaborator": 650,
        "Sector": 1000,
        "Affiliation": 1900,
    },
    edge_color="#555555",
    edge_width=1.4,
    label_font_size=8,
    edge_labels=True,
    edge_label_font_size=6,
    figsize=(13, 9),
    title="Collaborator network (customized)",
    output=OUTPUT_DIR / "02_custom.png",
)
fig

### 6c. Filtered sub-graph

Use `filter_sector` / `filter_affiliation` to restrict the rendered graph to collaborators (and their immediate neighbors) in given sector(s) and/or affiliation(s).

In [ ]:
focus_sector = df["sector"].mode().iat[0]
fig = draw_graph(
    g,
    layout="shell",
    figsize=(10, 7),
    title=f"Collaborators in sector: {focus_sector}",
    filter_sector=focus_sector,
    edge_labels=True,
    output=OUTPUT_DIR / f"03_filter_{focus_sector.replace(' ', '_')}.png",
)
fig

## 7. Visualize in Kuzu Browser / Bloom

Paste the snippet below into Kuzu Browser to see everything at once:

In [ ]:
print(get_example("everything"))

In [ ]:
print(BLOOM_PERSPECTIVE_HINT)

## 8. (Optional) Wipe the graph

Uncomment the cell below to remove every node and relationship in the database. Use with care.

In [ ]:
# with Ingestor(
#     settings.uri, settings.user, settings.password, settings.database
# ) as ing:
#     ing.clear()
# print("Database cleared.")

### Query examples

Named Cypher examples were removed in the embedded Kuzu version. Use the web Graph/Map views or service helpers instead.
